**Run `harvest_orig.ipynb` first.** This notebook needs that combined CSV.

kW and kWh can be processed in either order.


## 2b. Harvest kW (15-minute average power)

Averages `3_phase_watt_total` onto 15-minute clock times. EPM7000 meters are converted from watts to kW; PQM2 is treated as already kW.

**You also need:** `harvest_meter_guide.csv` with `meter_name` and `meter_model`. Names must match the orig file **exactly** (including capitals) or values can be ~1,000× too big or small.

**You should get:** `../data/outputs/harvest_kw_YYMMDD-YYMMDD.csv` with `datetime`, `meter_name`, `mean_kw`.


### Enter input

- **`data_path`** — orig CSV. The date stamp (`250723-260508`) must match the file `harvest_orig` actually wrote.
- **`info_path`** — meter model guide.
- **`time_frame`** — `True` to keep only `start`…`end`; `False` for all rows.
- **`name`** — first part of the output filename.


In [ ]:
# directories
input_dir = '../data/extracts/'
output_dir = '../data/outputs/'

# data input csv
data_path = output_dir + 'harvest_orig_' + '250723-260508' + '.csv' #'harvest_orig_250723-251017.csv' #'harvest_orig_######-######.csv' # preprocessed harvest data from harvest_orig.ipynb

# guide of the meters with their meter model types
info_path = input_dir + 'harvest_meter_guide.csv' # given by Eileen

##########################################################################

# True if want to process data in a certain time frame, False to process all data
time_frame = False #True
start = None #'2025-09-07 00:00:00' # None if processing all data
end = None #'2025-09-09 23:59:59'# # None out if processing all data

##########################################################################

# processed kw data name
name = 'harvest' # for file naming

### Imports


In [2]:
import os, sys

sys.path.append(os.path.abspath('..'))
import modules.harvest_kw as hv_kw # import self defined module
import modules.file_naming as fn # import self defined module

### Load, average, and save

`load_data` → optional `filter_time_frame` → `process_kw_data` → save.


In [ ]:
# var for file naming
var = 'kw'

# load meter data and info from csvs into dataframes, cleaned and reformatted
df, info_df = hv_kw.load_data(data_path, info_path)

# process data for only dates ie: sept 7-9
if time_frame:
    df = hv_kw.filter_time_frame(df, start, end)

# processed kw data in dataframe
result_df = hv_kw.process_kw_data(df, info_df)

# create kw filename
kw_filename = output_dir + fn.make_filename(result_df, name, var, 'csv')

# create csv of processed data
result_df.to_csv(kw_filename, index=False)